In [2]:
"""
Log-HAR-RV: same idea as har_rv_forecast.py, but fit in log-variance space
instead of raw levels. Realized variance is heavy-tailed / roughly
log-normal, so a handful of huge spikes dominate a raw-level OLS fit
(hence the near-flat predictions and low R^2 you got). Logging compresses
the spikes and is the standard fix used in the HAR-RV literature.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

sq_returns = d['log_return']**2
rv_hourly = sq_returns.resample("1h").sum().dropna()

df = pd.DataFrame({"rv": rv_hourly})
df["rv_lag1"] = df["rv"].shift(1)
df["rv_daily"] = df["rv"].shift(1).rolling(24).mean()
df["rv_weekly"] = df["rv"].shift(1).rolling(24 * 7).mean()
df["rv_target"] = df["rv"].shift(-1)
df = df.dropna()

# ---- log-transform everything (small epsilon guards against log(0)) ------
eps = df["rv"][df["rv"] > 0].min() * 1e-3
for col in ["rv", "rv_lag1", "rv_daily", "rv_weekly", "rv_target"]:
    df[f"log_{col}"] = np.log(df[col] + eps)

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = ["log_rv_lag1", "log_rv_daily", "log_rv_weekly"]
X_train = sm.add_constant(train[feature_cols])
y_train = train["log_rv_target"]
X_test = sm.add_constant(test[feature_cols], has_constant="add")
y_test = test["log_rv_target"]

model = sm.OLS(y_train, X_train).fit()
print(model.summary())
print("\nFitted kernel weights (K), log space:")
print(model.params, "\n")

# ---- out-of-sample R^2, both in log space and back-transformed levels ----
test_pred_log = model.predict(X_test)
ss_res = ((y_test - test_pred_log) ** 2).sum()
ss_tot = ((y_test - y_test.mean()) ** 2).sum()
oos_r2_log = 1 - ss_res / ss_tot

# back-transform to compare against the original raw-level plot
test_pred_level = np.exp(test_pred_log) - eps
y_test_level = np.exp(y_test) - eps
ss_res_lvl = ((y_test_level - test_pred_level) ** 2).sum()
ss_tot_lvl = ((y_test_level - y_test_level.mean()) ** 2).sum()
oos_r2_level = 1 - ss_res_lvl / ss_tot_lvl

print(f"In-sample R^2 (log space):      {model.rsquared:.4f}")
print(f"Out-of-sample R^2 (log space):  {oos_r2_log:.4f}")
print(f"Out-of-sample R^2 (raw levels): {oos_r2_level:.4f}")
print("(compare the raw-levels number directly against your previous run)\n")

# ---- plot: back-transformed predictions vs actual, same scale as before --
fig, ax = plt.subplots(figsize=(10, 4))
y_test_level.plot(ax=ax, label="actual next-hour RV", color="steelblue", lw=1)
test_pred_level.plot(ax=ax, label="predicted next-hour RV (log-HAR)", color="red", lw=1, ls="--")
ax.set_title("Log-HAR-RV forecast vs actual (out-of-sample, back-transformed)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("har_rv_log_forecast.png", dpi=150)
plt.close()

print("Compare model.params here vs. the raw-level run: coefficients are")
print("now on log-RV, so read them as 'a 1% rise in this lag's variance")
print("is associated with a beta% rise in next-hour variance' -- an")
print("elasticity, not a direct level effect.")

NameError: name 'd' is not defined

In [ ]:
"""
HAR-Return: same HAR kernel structure as the log-HAR-RV cell above (lag1 +
rolling-24h "daily" + rolling-168h "weekly" features, OLS, 80/20 split),
but fit on signed hourly log-returns instead of squared returns. RV only
ever tells you the expected *size* of the next move; regressing on the
raw (signed) return lets the same lag/daily/weekly kernel also pick up
directional persistence (momentum) or mean-reversion, so the forecast
carries both magnitude and sign.

No log-transform here: unlike RV, returns are signed, so log(ret) isn't
defined. Returns are already far less heavy-tailed than squared returns,
so raw-level OLS is the natural fit (this is also why the RV cell had to
log-transform and this one doesn't).
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

ret_hourly = log_return.resample("1h").sum().dropna()

df_r = pd.DataFrame({"ret": ret_hourly})
df_r["ret_lag1"] = df_r["ret"].shift(1)
df_r["ret_daily"] = df_r["ret"].shift(1).rolling(24).mean()
df_r["ret_weekly"] = df_r["ret"].shift(1).rolling(24 * 7).mean()
df_r["ret_target"] = df_r["ret"].shift(-1)
df_r = df_r.dropna()

split_r = int(len(df_r) * 0.8)
train_r, test_r = df_r.iloc[:split_r], df_r.iloc[split_r:]

feature_cols_r = ["ret_lag1", "ret_daily", "ret_weekly"]
X_train_r = sm.add_constant(train_r[feature_cols_r])
y_train_r = train_r["ret_target"]
X_test_r = sm.add_constant(test_r[feature_cols_r], has_constant="add")
y_test_r = test_r["ret_target"]

model_r = sm.OLS(y_train_r, X_train_r).fit()
print(model_r.summary())
print("\nFitted kernel weights (K), return space:")
print(model_r.params, "\n")

# ---- out-of-sample R^2 -----------------------------------------------
test_pred_r = model_r.predict(X_test_r)
ss_res_r = ((y_test_r - test_pred_r) ** 2).sum()
ss_tot_r = ((y_test_r - y_test_r.mean()) ** 2).sum()
oos_r2_r = 1 - ss_res_r / ss_tot_r

print(f"In-sample R^2:      {model_r.rsquared:.4f}")
print(f"Out-of-sample R^2:  {oos_r2_r:.4f}\n")

# ---- directional accuracy: the thing RV could never give you ---------
# hit rate = how often sign(predicted next-hour return) matches
# sign(actual next-hour return). 0.5 is coin-flip; compare against the
# naive "predict same sign as last hour" baseline too.
pred_sign = np.sign(test_pred_r)
actual_sign = np.sign(y_test_r)
nonzero = actual_sign != 0
hit_rate = (pred_sign[nonzero] == actual_sign[nonzero]).mean()

naive_sign = np.sign(test_r["ret_lag1"])
naive_hit_rate = (naive_sign[nonzero] == actual_sign[nonzero]).mean()

print(f"Directional hit rate (HAR-Return):      {hit_rate:.4f}")
print(f"Directional hit rate (naive, sign(lag1)): {naive_hit_rate:.4f}")
print("(both vs. 0.5 coin-flip baseline)\n")

# ---- plot: predicted vs actual next-hour return, out-of-sample -------
fig, ax = plt.subplots(figsize=(10, 4))
y_test_r.plot(ax=ax, label="actual next-hour return", color="steelblue", lw=1)
test_pred_r.plot(ax=ax, label="predicted next-hour return (HAR-Return)", color="red", lw=1, ls="--")
ax.axhline(0, color="black", lw=0.5)
ax.set_title("HAR-Return forecast vs actual (out-of-sample)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("har_ret_forecast.png", dpi=150)
plt.close()

print("Compare model_r.params here vs. the RV cell: coefficients are on")
print("raw signed returns, so a positive ret_lag1 coefficient means")
print("momentum (last hour's direction tends to continue) and a negative")
print("one means mean-reversion.")
